# Chroma DB ZIP 생성 및 다운로드

현재 Colab 런타임에 남아 있는 실제 Chroma DB를 자동으로 찾아 ZIP으로 압축하고, 압축 파일을 검증한 뒤 다운로드합니다. **DB를 생성한 원래 런타임에서 위에서부터 순서대로 실행하세요.**

In [ ]:
# 1. 실제 Chroma DB 폴더 자동 탐색
from pathlib import Path

# 자동 탐색이 안 될 때만 실제 경로를 문자열로 넣으세요. 예: '/content/.../kosis_meta_chroma_holdout8_v8'
MANUAL_DB_DIR = None
CONTENT = Path('/content')

if MANUAL_DB_DIR:
    candidates = [Path(MANUAL_DB_DIR)]
else:
    # 실제 Chroma 영속 DB에는 chroma.sqlite3가 들어 있습니다. manifest만 있는 결과 폴더는 제외합니다.
    candidates = sorted(
        {p.parent for p in CONTENT.rglob('chroma.sqlite3') if p.is_file()},
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

if not candidates:
    manifests = sorted(CONTENT.rglob('chroma_manifest.json'))
    print('발견된 manifest:', *[str(p) for p in manifests], sep='\n- ')
    raise FileNotFoundError(
        '현재 런타임에 실제 Chroma DB(chroma.sqlite3)가 없습니다. '
        'DB를 만든 원래 Colab 런타임에서 실행하세요. 런타임이 재시작됐다면 인덱스를 다시 생성해야 합니다.'
    )

print('발견된 실제 Chroma DB 후보:')
for i, path in enumerate(candidates, 1):
    print(f'{i}. {path}')

# 후보가 여러 개면 가장 최근 수정된 DB를 선택합니다. 다른 DB라면 MANUAL_DB_DIR에 경로를 지정하세요.
DB_DIR = candidates[0]
DB_NAME = DB_DIR.name

files_in_db = [p for p in DB_DIR.rglob('*') if p.is_file()]
if not files_in_db:
    raise RuntimeError(f'Chroma DB 폴더가 비어 있습니다: {DB_DIR}')

print('Chroma DB:', DB_DIR)
print('파일 수:', len(files_in_db))
print('원본 크기:', f'{sum(p.stat().st_size for p in files_in_db) / 1024**2:,.2f} MB')

In [ ]:
# 2. ZIP 생성
import shutil

ZIP_BASE = Path('/content') / DB_NAME
ZIP_PATH = Path(shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=DB_DIR.parent,
    base_dir=DB_DIR.name,
))

print('생성 완료:', ZIP_PATH)
print('ZIP 크기:', f'{ZIP_PATH.stat().st_size / 1024**2:,.2f} MB')

In [ ]:
# 3. ZIP 손상 및 누락 여부 확인
import zipfile

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    broken = zf.testzip()
    archived_files = [n for n in zf.namelist() if not n.endswith('/')]

if broken is not None:
    raise RuntimeError(f'ZIP 파일이 손상되었습니다: {broken}')
if len(archived_files) != len(files_in_db):
    raise RuntimeError(
        f'파일 수가 다릅니다: 원본 {len(files_in_db)}개, ZIP {len(archived_files)}개'
    )

print('검증 통과')
print('ZIP 내부 파일 수:', len(archived_files))
print('다운로드 대상:', ZIP_PATH)

In [ ]:
# 4. PC로 다운로드
from google.colab import files

files.download(str(ZIP_PATH))